# facenet_cpp — Thrust device (GPU) check: eval forward parity + cuDNN
The device engine (`dface_ops.hpp`/`dface.hpp`) is CPU-thrust-verified (1.42e-07) and nvcc-compiles;
this runs it on a real GPU. **Runtime → GPU (T4)**, then Run all.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf facenet_cpp
!git clone -q https://github.com/yomei-o/facenet_cpp.git
%cd /content/facenet_cpp
!pip -q install facenet-pytorch


### Export refs (fused + unfused weights + fp32 parity embedding)


In [ ]:
!python pure/ref/export_facenet.py 160   # -> pure/ref/data_net/{manifest_unfused,weights_unfused}.bin + ref/{input,embed}.bin


### Locate cuDNN (for step 2) — auto


In [ ]:
import os, glob
inc=lib=None
try:
    import nvidia.cudnn; d=os.path.dirname(nvidia.cudnn.__file__)
    if os.path.exists(d+"/include/cudnn.h"): inc,lib=d+"/include",d+"/lib"
except Exception: pass
if not inc:
    for h in ["/usr/include/cudnn.h"]+glob.glob("/usr/include/**/cudnn.h",recursive=True)+glob.glob("/usr/local/cuda*/include/cudnn.h"):
        if os.path.exists(h): inc,lib=os.path.dirname(h),"/usr/lib/x86_64-linux-gnu"; break
os.environ["CUDNN_INC"],os.environ["CUDNN_LIB"]=inc or "",lib or ""
print("CUDNN_INC =",inc,"\nCUDNN_LIB =",lib)


### 1. Device eval forward parity on GPU (dface_test) — expect MATCH ~1e-6


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/dface_test.cpp -o dface_gpu
!./dface_gpu pure/ref/data_net/ pure/ref/ 160


The device forward uses eval-mode BN (running stats), so it reproduces the facenet-pytorch reference embedding (CPU-thrust gave 1.42e-07).


### 2. cuDNN device path — dface_test with -DUSE_CUDNN


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUDNN -diag-suppress 550 \
      -I"$CUDNN_INC" -L"$CUDNN_LIB" -Ipure/third_party pure/dface_test.cpp -lcudnn -o dface_cudnn
!LD_LIBRARY_PATH="$CUDNN_LIB:$LD_LIBRARY_PATH" ./dface_cudnn pure/ref/data_net/ pure/ref/ 160


Both should print `backend: GPU (CUDA)` + MATCH. This is the only part of the facenet device engine not yet run on real GPU hardware.
